# MergeLane: code-reading model (free GPU)

This notebook fine-tunes a small code model (UniXcoder, 125M parameters) to read each PR's diff and predict trouble.

**How to run on Kaggle (free):**
1. Settings → Accelerator → **GPU T4 x2**, and Internet **on**.
2. Add Data → Upload → `mergelane_diffs.parquet`.
3. Run all. It takes about 45 minutes.
4. Download `code_scores.csv` from the Output panel and send it back.

In [ ]:
import glob, numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import roc_auc_score
path = (glob.glob('/kaggle/input/**/mergelane_diffs.parquet', recursive=True) + glob.glob('mergelane_diffs.parquet'))[0]
df = pd.read_parquet(path)
SMOKE = False  # True = quick run on a tiny sample
if SMOKE: df = pd.concat([g.sample(min(len(g), 48), random_state=0) for _, g in df.groupby('split')]); print(df.split.value_counts())
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev, torch.cuda.device_count())

In [ ]:
NAME, MAXLEN, BS, EPOCHS, LR = 'microsoft/unixcoder-base', 512, 16, 1, 2e-5
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForSequenceClassification.from_pretrained(NAME, num_labels=1).to(dev)
if torch.cuda.device_count() > 1: model = torch.nn.DataParallel(model)
class DS(Dataset):
    def __init__(self, d): self.t, self.y = d.text.tolist(), d.label.astype('float32').tolist()
    def __len__(self): return len(self.t)
    def __getitem__(self, i): return self.t[i], self.y[i]
def collate(b):
    t, y = zip(*b); e = tok(list(t), truncation=True, max_length=MAXLEN, padding=True, return_tensors='pt')
    return e, torch.tensor(y)
train = df[df.split == 'train'].sample(frac=1, random_state=0)
pos_w = torch.tensor((1 - train.label.mean()) / train.label.mean()).to(dev)

In [ ]:
dl = DataLoader(DS(train), batch_size=BS, shuffle=True, collate_fn=collate, num_workers=2)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sch = get_linear_schedule_with_warmup(opt, int(0.06 * len(dl) * EPOCHS), len(dl) * EPOCHS)
scaler = torch.amp.GradScaler('cuda', enabled=(dev == 'cuda')); lossf = torch.nn.BCEWithLogitsLoss(pos_weight=pos_w)
model.train()
for ep in range(EPOCHS):
    for step, (e, y) in enumerate(dl):
        e = {k: v.to(dev) for k, v in e.items()}
        with torch.autocast('cuda', dtype=torch.float16, enabled=(dev == 'cuda')):
            loss = lossf(model(**e).logits.squeeze(-1).float(), y.to(dev))
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sch.step()
        if step % 200 == 0: print(ep, step, len(dl), round(loss.item(), 4), flush=True)

In [ ]:
model.eval(); rest = df[df.split != 'train']
scores = []
with torch.no_grad():
    for e, _ in DataLoader(DS(rest), batch_size=64, collate_fn=collate, num_workers=2):
        e = {k: v.to(dev) for k, v in e.items()}
        with torch.autocast('cuda', dtype=torch.float16, enabled=(dev == 'cuda')):
            scores.append(torch.sigmoid(model(**e).logits.squeeze(-1).float()).cpu().numpy())
rest = rest.assign(score=np.concatenate(scores))
t, h = rest[rest.split == 'test'], rest[rest.split == 'held']
print('test AUC', round(roc_auc_score(t.label, t.score), 3), '| real vs false', round(roc_auc_score(h.verdict == 'yes', h.score), 3))
rest[['pr_id', 'split', 'score']].to_csv('/kaggle/working/code_scores.csv', index=False)